In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import kagglehub

import os
import re
import pathlib
import pandas as pd
from PIL import Image
import numpy as np
import reverse_geocoder as rg
import pycountry
import pycountry_convert as pc
import plotly.express as px
import plotly.offline as pyo


import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot

from math import ceil, sqrt

# Import inbuilt libraries
import os, shutil, pathlib
from math import sqrt, ceil

# Import necessary libraries for data analysis
import numpy as np
import pandas as pd

# Import necessary libraries for data visualization
from plotly.offline import init_notebook_mode, iplot
import plotly.graph_objects as go
import plotly.offline as pyo
import plotly.express as px

# Initialize Plotly for Jupyter notebook
init_notebook_mode(connected=True)

/home/pedrosalgado22/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IMAGES_PATH = Path() / "images" / "classification"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)




plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [3]:
DATA_PATH = Path("data/images/streetview_images")

if not any(DATA_PATH.iterdir()):
    path = kagglehub.dataset_download(
        "sylshaw/streetview-by-country",
        output_dir=str(DATA_PATH)
    )
else:
    path = str(DATA_PATH)

print("Path to dataset:", path)

Path to dataset: data/images/streetview_images


In [4]:
geo_data_dir = pathlib.Path(path)


def extract_metadata_from_folder(path):
    metadata = []
    for file_path in path.glob('*'):   # loops through every item inside path
        width, height = Image.open(file_path).size

        lat_str, lon_str, heading_str = file_path.stem.split('_')
        latitude = float(lat_str.lstrip('n').replace('d', '.'))
        if lat_str.startswith('n'):
            latitude = -latitude
        longitude = float(lon_str.lstrip('n').replace('d', '.'))
        if lon_str.startswith('n'):
            longitude = -longitude
        heading = int(heading_str[1:])
        if heading_str.startswith('n'):
            heading = -heading

        metadata.append({
            'country_code': path.name,
            'image_name': file_path.name,
            'width': width,
            'height': height,
            'size': file_path.stat().st_size,
            'path': file_path,
            'latitude': latitude,
            'longitude': longitude,
            'heading': heading
        })
    return metadata



In [5]:
# List all country folders
country_folders = [f for f in os.listdir(geo_data_dir) if os.path.isdir(os.path.join(geo_data_dir, f))]

# Initialize an empty list to store metadata
all_metadata = []

# Iterate through country folders and extract metadata
for country_name in country_folders:
    metadata = extract_metadata_from_folder(pathlib.Path(os.path.join(geo_data_dir, country_name)))
    all_metadata.extend(metadata)

# Create a Pandas DataFrame from the metadata
df_geo_data = pd.DataFrame(all_metadata)

def get_country_codes(df):
    coords = list(zip(df['latitude'], df['longitude']))
    return [r['cc'] for r in rg.search(coords, mode=1)]

def get_country(codes):
    return [pycountry.countries.get(alpha_2=c).name if pycountry.countries.get(alpha_2=c) else c for c in codes]

def get_continent(codes):
    result = []
    for c in codes:
        try:
            result.append(pc.convert_continent_code_to_continent_name(pc.country_alpha2_to_continent_code(c)))
        except KeyError:
            result.append(None)
    return result

coord_codes = get_country_codes(df_geo_data)
df_geo_data['country'] = get_country(coord_codes)
df_geo_data['continent'] = get_continent(coord_codes)

df_data_distribution = df_geo_data.groupby('country')['image_name'].count().reset_index().rename(columns={'image_name': 'frequency'})

df_geo_data.head()

Loading formatted geocoded file...


,country_code,image_name,width,height,size,path,latitude,longitude,heading,country,continent
0,AD,42d436079_1d473612_h213.jpg,640,640,71291,data/images/streetview_images/AD/42d436079_1d4...,42.436079,1.473612,213,Andorra,Europe
1,AD,42d436223_1d521434_h252.jpg,640,640,86050,data/images/streetview_images/AD/42d436223_1d5...,42.436223,1.521434,252,Andorra,Europe
2,AD,42d437356_1d476157_h207.jpg,640,640,75563,data/images/streetview_images/AD/42d437356_1d4...,42.437356,1.476157,207,Andorra,Europe
3,AD,42d438534_1d521492_h237.jpg,640,640,99736,data/images/streetview_images/AD/42d438534_1d5...,42.438534,1.521492,237,Andorra,Europe
4,AD,42d439193_1d490868_h254.jpg,640,640,70899,data/images/streetview_images/AD/42d439193_1d4...,42.439193,1.490868,254,Andorra,Europe


In [6]:
df_geo_data['coord_country_code'] = coord_codes

mismatches = df_geo_data[df_geo_data['coord_country_code'] != df_geo_data['country_code']]

print(f"{len(mismatches)} mismatches")

mismatches[['image_name', 'country_code', 'coord_country_code', 'country']]

print(df_geo_data[['country', 'continent']].isna().sum())

3572 mismatches
country      0
continent    2
dtype: int64


In [7]:
# select a random country
random_country = df_geo_data.sample(1)

with Image.open(random_country.path.values[0]) as img:
    fig = go.Figure()
    fig.add_trace(go.Image(z=img))
    fig.update_layout(
        title="Image from the training dataset",
        template="plotly_white",
        width=900,
        height=450
    )
    
    fig.update_xaxes(title_text="Width")
    fig.update_yaxes(title_text="Height")
    
    fig.add_annotation(
        x=0.5,
        y=1.1,
        xref="paper",
        yref="paper",
        text="Country : {}".format(random_country.country.values[0]),
        showarrow=False
    )
    
    fig.show()

In [8]:
print("The total number of images in the dataset are {:,} images.".format(df_data_distribution.frequency.sum()))
print("The average number of images per country is {:,} images.".format(int(df_data_distribution.frequency.mean())))
print("The median number of images per country is {:,} images.".format(int(df_data_distribution.frequency.median())))
print("25% of the countries have {:,} images or less, while 75% of the countries have {:,} images or more.".format(int(df_data_distribution.frequency.quantile(0.25)), int(df_data_distribution.frequency.quantile(0.75))))
print("The maximum and minimum number of images per country are {:,} and {:,} images respectively.".format(df_data_distribution.frequency.max(), df_data_distribution.frequency.min()))

The total number of images in the dataset are 107,269 images.
The average number of images per country is 719 images.
The median number of images per country is 976 images.
25% of the countries have 111 images or less, while 75% of the countries have 1,000 images or more.
The maximum and minimum number of images per country are 1,408 and 1 images respectively.


In [9]:
# Function to plot heatmap
def plot_heatmap(df, title_text, color_threshold=0.90):
    # Shaping Data to fit graph by finding the max size of the grid to fit all the images
    grid_size = ceil(sqrt(df.shape[0]))**2
    
    pad = '-'
    n_labels = df.country.to_list() + ([pad] * (grid_size - df.shape[0]))
    n_samples = df.frequency.to_list() + ([np.nan] * (grid_size - df.shape[0]))
    
    samples = np.split(np.array(n_samples), sqrt(grid_size))
    
    fig = go.Figure(data=go.Heatmap(
        z=samples,
        x0=0,
        dx=1,
        y0=0,
        dy=1,
        colorscale='Emrld',
        text=n_labels,
        hoverinfo='text',
    ))
    
    # Add labels within the heatmap
    for i, country in enumerate(df['country']):
        x_pos = i % int(sqrt(grid_size))
        y_pos = i // int(sqrt(grid_size))
        fig.add_annotation(
            x=x_pos,  # Center the label within the grid cell
            y=y_pos,
            text= "{}<br> Count: {}".format(n_labels[i].replace(' ', '<br>'), n_samples[i]),
            showarrow=False,
            font=dict(
                color='#474747' if n_samples[i] < df.frequency.quantile(color_threshold) else 'white',
                size=10),
        )
    
    # hide axis labels
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False, autorange='reversed')
    
    # change the dimensions of the figure
    fig.update_layout(
        title=title_text,
        height=1200,
        width=1200
    )
    fig = add_plot_styling(fig)
    
    # Use iplot to display the graph in the Jupyter Notebook
    pyo.iplot(fig)

In [10]:
# Function to add custom styling to the plot
def add_plot_styling(fig):
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0.05)',  # Set background color to transparent
    )
    return fig



In [11]:
plot_heatmap(df_data_distribution.sort_values(by='frequency', ascending=True), "Images per Country", color_threshold=0.98)

In [12]:
df_geo_data.keys()

Index(['country_code', 'image_name', 'width', 'height', 'size', 'path',
       'latitude', 'longitude', 'heading', 'country', 'continent',
       'coord_country_code'],
      dtype='str')

In [13]:
# outlier removal
filtered_countries = df_data_distribution[(df_data_distribution.frequency >= 110) & (df_data_distribution.frequency <= 10000) ].country.unique()

df_filtered_geo_data = df_geo_data[df_geo_data.country.isin(filtered_countries)]


In [14]:
plot_heatmap(df_filtered_geo_data.groupby('country')['image_name'].count().reset_index().rename(columns={'image_name': 'frequency'}).sort_values(by='frequency', ascending=True), "Images per Country", color_threshold=0.98)

In [15]:
from hsv import color_histogram
from multiprocessing import Pool

with Pool() as pool:
    results = pool.map(color_histogram, df_filtered_geo_data["path"])

color_cols = pd.DataFrame(results, columns=[f'hsv_bin_{i}' for i in range(48)])
df_filtered_geo_data = pd.concat([df_filtered_geo_data.reset_index(drop=True), color_cols], axis=1)

In [16]:
df_filtered_geo_data.head()

,country_code,image_name,width,height,size,path,latitude,longitude,heading,country,...,hsv_bin_38,hsv_bin_39,hsv_bin_40,hsv_bin_41,hsv_bin_42,hsv_bin_43,hsv_bin_44,hsv_bin_45,hsv_bin_46,hsv_bin_47
0,AD,42d436079_1d473612_h213.jpg,640,640,71291,data/images/streetview_images/AD/42d436079_1d4...,42.436079,1.473612,213,Andorra,...,0.069397,0.208933,0.134873,0.027637,0.021265,0.113418,0.146689,0.052222,0.011062,0.002815
1,AD,42d436223_1d521434_h252.jpg,640,640,86050,data/images/streetview_images/AD/42d436223_1d5...,42.436223,1.521434,252,Andorra,...,0.035520,0.082715,0.104885,0.107693,0.135332,0.129006,0.101323,0.036992,0.012727,0.008496
2,AD,42d437356_1d476157_h207.jpg,640,640,75563,data/images/streetview_images/AD/42d437356_1d4...,42.437356,1.476157,207,Andorra,...,0.136467,0.224182,0.092813,0.019526,0.005422,0.032190,0.146052,0.096953,0.018948,0.005381
3,AD,42d438534_1d521492_h237.jpg,640,640,99736,data/images/streetview_images/AD/42d438534_1d5...,42.438534,1.521492,237,Andorra,...,0.051079,0.131533,0.161086,0.055222,0.035322,0.061565,0.067944,0.064136,0.025212,0.015088
4,AD,42d439193_1d490868_h254.jpg,640,640,70899,data/images/streetview_images/AD/42d439193_1d4...,42.439193,1.490868,254,Andorra,...,0.067200,0.035320,0.068201,0.174233,0.161079,0.080024,0.052656,0.143879,0.045042,0.002725
